In [328]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

In [329]:
df = pd.read_csv("../data/processed/customer_data_cleaned.csv")

df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Total_Spending
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,0,0,0,0,0,0,3,11,1,1617
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,0,0,0,0,0,0,3,11,0,27
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,0,0,0,0,0,0,3,11,0,776
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,0,0,0,0,0,0,3,11,0,53
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,0,0,0,0,0,0,3,11,0,422


# Customer Features

Customer features describe the demographic and household characteristics of each customer. These engineered variables provide additional context that may influence purchasing behaviour and improve customer segmentation.

### Age

Age is derived from the customer's year of birth and represents a more intuitive demographic feature than the original birth year.

In [330]:
current_year = datetime.now().year

df["Age"] = current_year - df["Year_Birth"]

df[["Year_Birth", "Age"]].head()

,Year_Birth,Age
0,1957,69
1,1954,72
2,1965,61
3,1984,42
4,1981,45


In [331]:
df["Age"].describe()

count    2240.000000
mean       57.194196
std        11.984069
min        30.000000
25%        49.000000
50%        56.000000
75%        67.000000
max       133.000000
Name: Age, dtype: float64

### Customer Tenure

Customer tenure measures the number of years since a customer joined the company and serves as a proxy for customer loyalty and relationship duration.

In [332]:
df["Dt_Customer"] = pd.to_datetime(
    df["Dt_Customer"],
    format="%d-%m-%Y"
)

In [333]:
current_date = pd.Timestamp.today()

df["Customer_Tenure"] = (
    (current_date - df["Dt_Customer"]).dt.days / 365
).round(1)

df[["Dt_Customer", "Customer_Tenure"]].head()

,Dt_Customer,Customer_Tenure
0,2012-09-04,13.9
1,2014-03-08,12.4
2,2013-08-21,12.9
3,2014-02-10,12.5
4,2014-01-19,12.5


In [334]:
df["Customer_Tenure"].describe()

count    2240.000000
mean       13.044554
std         0.553858
min        12.100000
25%        12.600000
50%        13.050000
75%        13.500000
max        14.000000
Name: Customer_Tenure, dtype: float64

### Children

The **Children** feature represents the total number of children in a customer's household by combining the number of young children and teenagers. Household composition can influence purchasing behaviour, spending priorities, and product preferences.

In [335]:
# Create total number of children in the household
df["Children"] = df["Kidhome"] + df["Teenhome"]

# Preview the new feature
df[["Kidhome", "Teenhome", "Children"]].head()

,Kidhome,Teenhome,Children
0,0,0,0
1,1,1,2
2,0,0,0
3,1,0,1
4,1,0,1


In [336]:
df["Children"].describe()

count    2240.000000
mean        0.950446
std         0.751803
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: Children, dtype: float64

### Household Type

The **Household_Type** feature consolidates marital status into broader household categories. Grouping customers into **Partner** and **Alone** reduces category sparsity while preserving the household structure most relevant for customer segmentation.

In [337]:
household_mapping = {
    "Married": "Partner",
    "Together": "Partner",
    "Single": "Alone",
    "Divorced": "Alone",
    "Widow": "Alone",
    "Alone": "Alone",
    "YOLO": "Alone",
    "Absurd": "Alone"
}

df["Household_Type"] = df["Marital_Status"].map(household_mapping)

df[["Marital_Status", "Household_Type"]].head()

,Marital_Status,Household_Type
0,Single,Alone
1,Single,Alone
2,Together,Partner
3,Together,Partner
4,Married,Partner


In [338]:
df["Household_Type"].value_counts()

Household_Type
Partner    1444
Alone       796
Name: count, dtype: int64

# Behavioural Features

Behavioural features capture how customers interact with the business through their purchasing activity, spending patterns, and marketing engagement. These engineered variables provide a richer representation of customer behaviour than the original raw features and form the foundation for customer segmentation.

### Total Spending

The **Total_Spending** feature represents the customer's overall monetary contribution by aggregating spending across all product categories. This provides a single measure of customer value and serves as the **Monetary (M)** component in the RFM framework.

In [339]:
spending_columns = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["Total_Spending"] = df[spending_columns].sum(axis=1)

df[["Total_Spending"]].head()

,Total_Spending
0,1617
1,27
2,776
3,53
4,422


In [340]:
df["Total_Spending"].describe()

count    2240.000000
mean      605.798214
std       602.249288
min         5.000000
25%        68.750000
50%       396.000000
75%      1045.500000
max      2525.000000
Name: Total_Spending, dtype: float64

### Purchase Frequency

The **Purchase_Frequency** feature represents the total number of purchases made across all sales channels. Unlike spending, this feature measures customer purchasing activity and will serve as the **Frequency (F)** component in the RFM framework.

In [341]:
purchase_columns = [
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases"
]

df["Purchase_Frequency"] = df[purchase_columns].sum(axis=1)

df[["Purchase_Frequency"]].head()

,Purchase_Frequency
0,22
1,4
2,20
3,6
4,14


In [342]:
df["Purchase_Frequency"].describe()

count    2240.000000
mean       12.537054
std         7.205741
min         0.000000
25%         6.000000
50%        12.000000
75%        18.000000
max        32.000000
Name: Purchase_Frequency, dtype: float64

### Campaign Engagement

The **Campaign_Engagement** feature measures how frequently a customer has responded positively to marketing campaigns. Customers with higher engagement may be more receptive to future promotional activities and loyalty programs.

In [343]:
campaign_columns = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
    "Response"
]

df["Campaign_Engagement"] = df[campaign_columns].sum(axis=1)

df[["Campaign_Engagement"]].head()

,Campaign_Engagement
0,1
1,0
2,0
3,0
4,0


In [344]:
df["Campaign_Engagement"].value_counts().sort_index()

Campaign_Engagement
0    1631
1     370
2     142
3      51
4      36
5      10
Name: count, dtype: int64

In [345]:
behaviour_summary = pd.DataFrame({
    "Feature": [
        "Total_Spending",
        "Purchase_Frequency",
        "Campaign_Engagement"
    ],
    "Minimum": [
        df["Total_Spending"].min(),
        df["Purchase_Frequency"].min(),
        df["Campaign_Engagement"].min()
    ],
    "Maximum": [
        df["Total_Spending"].max(),
        df["Purchase_Frequency"].max(),
        df["Campaign_Engagement"].max()
    ],
    "Mean": [
        round(df["Total_Spending"].mean(), 2),
        round(df["Purchase_Frequency"].mean(), 2),
        round(df["Campaign_Engagement"].mean(), 2)
    ]
})

behaviour_summary

,Feature,Minimum,Maximum,Mean
0,Total_Spending,5,2525,605.80
1,Purchase_Frequency,0,32,12.54
2,Campaign_Engagement,0,5,0.45


# Marketing Features

Marketing features are engineered to capture customer engagement, purchasing behaviour, and overall customer value. These variables provide meaningful business representations that will be used for customer segmentation and cluster profiling.

### RFM Framework

RFM (Recency, Frequency, Monetary) is a widely used customer segmentation framework that evaluates customer value based on three key dimensions:

- **Recency (R):** How recently a customer made a purchase.
- **Frequency (F):** How often a customer makes purchases.
- **Monetary (M):** How much a customer spends.

In this project, these dimensions are represented using previously engineered behavioural features:

| RFM Component | Feature Used |
|--------------|--------------|
| **Recency (R)** | `Recency` |
| **Frequency (F)** | `Purchase_Frequency` |
| **Monetary (M)** | `Total_Spending` |

These variables will serve as key inputs for customer segmentation and cluster profiling in the subsequent modelling phase.

In [346]:
rfm_df = df[
    [
        "Recency",
        "Purchase_Frequency",
        "Total_Spending"
    ]
].copy()

rfm_df.head()

,Recency,Purchase_Frequency,Total_Spending
0,58,22,1617
1,38,4,27
2,26,20,776
3,26,6,53
4,94,14,422


### Customer Value Score

The **Customer Value Score** combines customer spending, purchasing frequency, and campaign engagement into a single normalized metric. By bringing these behavioural dimensions onto the same scale, the score provides an overall indication of a customer's relative value to the business.

Unlike the individual RFM metrics, this composite score offers a simplified measure that can be used to compare customers, support cluster interpretation, and identify high-value customer segments.

In [347]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_features = scaler.fit_transform(
    df[
        [
            "Total_Spending",
            "Purchase_Frequency",
            "Campaign_Engagement"
        ]
    ]
)

scaled_df = pd.DataFrame(
    scaled_features,
    columns=[
        "Spending",
        "Frequency",
        "Engagement"
    ]
)

In [348]:
df["Customer_Value_Score"] = (
    scaled_df["Spending"] +
    scaled_df["Frequency"] +
    scaled_df["Engagement"]
) / 3

In [349]:
df["Customer_Value_Score"].describe()

count    2240.000000
mean        0.239857
std         0.179746
min         0.000000
25%         0.073479
50%         0.199735
75%         0.368089
max         0.891534
Name: Customer_Value_Score, dtype: float64

- The Customer Value Score ranges between **0** and **1**, where higher values represent customers with stronger overall purchasing behaviour and marketing engagement.
- Customers with higher scores generally exhibit greater spending, purchase more frequently, and respond more positively to marketing campaigns.
- This feature provides a concise measure of customer value that complements the RFM framework and will support customer segmentation and cluster profiling.

### Deal Engagement

The **Deal_Engagement** feature represents the number of purchases made using promotional deals or discounts. It provides an indication of how frequently a customer responds to price-based promotions and helps identify customers who are more receptive to discount-driven marketing strategies.

Unlike a ratio-based metric, this feature directly reflects promotional purchasing activity without introducing ambiguity in interpretation.

In [350]:
# Create Deal Engagement feature
df["Deal_Engagement"] = df["NumDealsPurchases"]

# Preview the feature
df[["NumDealsPurchases", "Deal_Engagement"]].head()

,NumDealsPurchases,Deal_Engagement
0,3,3
1,2,2
2,1,1
3,2,2
4,5,5


In [351]:
df["Deal_Engagement"].describe()

count    2240.000000
mean        2.325000
std         1.932238
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        15.000000
Name: Deal_Engagement, dtype: float64

In [352]:
df["Deal_Engagement"].value_counts().sort_index()

Deal_Engagement
0      46
1     970
2     497
3     297
4     189
5      94
6      61
7      40
8      14
9       8
10      5
11      5
12      4
13      3
15      7
Name: count, dtype: int64

- Most customers exhibit relatively low Deal Engagement, with the majority making between **1 and 3 purchases** using promotional offers.
- Only a small proportion of customers demonstrate very high deal engagement (greater than 7), indicating that highly promotion-driven customers represent a niche segment.
- This feature helps identify price-sensitive customers who are more likely to respond to discount-based marketing campaigns and promotional offers.

# Data Preprocessing

The engineered dataset is prepared for clustering by removing redundant variables, handling missing values, encoding categorical features, and scaling numerical variables. These preprocessing steps ensure that all features are suitable for distance-based clustering algorithms such as K-Means.

In [353]:
columns_to_drop = [
    "ID",
    "Year_Birth",
    "Kidhome",
    "Teenhome",
    "Dt_Customer",
    "Marital_Status",
    "NumDealsPurchases",
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
    "Z_CostContact",
    "Z_Revenue",
    "Response"
]

df.drop(columns=columns_to_drop, inplace=True)

df.head()

,Education,Income,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumWebPurchases,...,Complain,Total_Spending,Age,Customer_Tenure,Children,Household_Type,Purchase_Frequency,Campaign_Engagement,Customer_Value_Score,Deal_Engagement
0,Graduation,58138.0,58,635,88,546,172,88,88,8,...,0,1617,69,13.9,0,Alone,22,1,0.509061,3
1,Graduation,46344.0,38,11,1,6,2,1,6,1,...,0,27,72,12.4,2,Alone,4,0,0.044577,2
2,Graduation,71613.0,26,426,49,127,111,21,42,8,...,0,776,61,12.9,0,Partner,20,0,0.310317,1
3,Graduation,26646.0,26,11,4,20,10,3,5,2,...,0,53,42,12.5,1,Partner,6,0,0.068849,2
4,PhD,58293.0,94,173,43,118,46,27,15,5,...,0,422,45,12.5,1,Partner,14,0,0.200992,5


## 5.2 Handle Missing Values

Before preparing the dataset for clustering, missing values must be addressed to ensure that all observations can be processed by machine learning algorithms.

A data quality assessment performed earlier in the project identified that **Income** is the only feature containing missing values, accounting for less than 2% of the dataset. Given the small proportion of missing observations, median imputation is applied to preserve all customer records while minimizing the influence of extreme income values.

In [354]:
# Check missing values
df.isnull().sum()

Education                0
Income                  24
Recency                  0
MntWines                 0
MntFruits                0
MntMeatProducts          0
MntFishProducts          0
MntSweetProducts         0
MntGoldProds             0
NumWebPurchases          0
NumCatalogPurchases      0
NumStorePurchases        0
NumWebVisitsMonth        0
Complain                 0
Total_Spending           0
Age                      0
Customer_Tenure          0
Children                 0
Household_Type           0
Purchase_Frequency       0
Campaign_Engagement      0
Customer_Value_Score     0
Deal_Engagement          0
dtype: int64

In [355]:
df["Income"].dtype


dtype('float64')

In [356]:
df["Income"] = df["Income"].fillna(df["Income"].median())

In [357]:
df.isnull().sum()

Education               0
Income                  0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
Complain                0
Total_Spending          0
Age                     0
Customer_Tenure         0
Children                0
Household_Type          0
Purchase_Frequency      0
Campaign_Engagement     0
Customer_Value_Score    0
Deal_Engagement         0
dtype: int64

## 5.3 Encode Categorical Variables

Machine learning algorithms require numerical inputs. The remaining categorical variables are transformed using **one-hot encoding**, which converts each category into binary indicator variables.

Both **Education** and **Household_Type** are treated as nominal categorical variables. Although education has an inherent progression, the distance between educational levels is not strictly quantitative. One-hot encoding avoids introducing artificial numerical relationships that could influence the distance calculations used by the K-Means clustering algorithm.

In [358]:
df = pd.get_dummies(
    df,
    columns=["Education", "Household_Type"],
    drop_first=True,
    dtype=int
)

In [359]:
# Display the newly created encoded columns
encoded_columns = [col for col in df.columns if "Education_" in col or "Household_Type_" in col]

print(encoded_columns)

['Education_Basic', 'Education_Graduation', 'Education_Master', 'Education_PhD', 'Household_Type_Partner']


### Interpretation

The categorical variables were successfully converted into numerical representations using one-hot encoding. This transformation enables the K-Means algorithm to process categorical information without imposing artificial numerical ordering, ensuring that each category contributes independently during clustering.

## 5.4 Feature Scaling

K-Means clustering is a distance-based algorithm that is sensitive to differences in feature scales. To ensure that no individual feature disproportionately influences the clustering process, all numerical variables are standardized using the StandardScaler.

Standardization transforms each feature to have a mean of 0 and a standard deviation of 1, making the dataset suitable for distance-based machine learning algorithms.

In [360]:
# Save Customer Value Score separately for later cluster profiling
customer_value_score = df["Customer_Value_Score"].copy()

# Remove it from the modelling dataset
df_model = df.drop(columns=["Customer_Value_Score"])

In [361]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_data = scaler.fit_transform(df_model)

scaled_df = pd.DataFrame(
    scaled_data,
    columns=df_model.columns
)

scaled_df.head()

,Income,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumWebPurchases,NumCatalogPurchases,...,Customer_Tenure,Children,Purchase_Frequency,Campaign_Engagement,Deal_Engagement,Education_Basic,Education_Graduation,Education_Master,Education_PhD,Household_Type_Partner
0,0.235696,0.307039,0.983781,1.551577,1.679702,2.462147,1.476500,0.843207,1.409304,2.510890,...,1.544867,-1.264505,1.313544,0.621248,0.349414,-0.157171,0.993769,-0.444816,-0.526385,-1.346874
1,-0.235454,-0.383664,-0.870479,-0.636301,-0.713225,-0.650449,-0.631503,-0.729006,-1.110409,-0.568720,...,-1.164012,1.396361,-1.185022,-0.501912,-0.168236,-0.157171,0.993769,-0.444816,-0.526385,-1.346874
2,0.773999,-0.798086,0.362723,0.570804,-0.177032,1.345274,-0.146905,-0.038766,1.409304,-0.226541,...,-0.261052,-1.264505,1.035926,-0.501912,-0.685887,-0.157171,0.993769,-0.444816,-0.526385,0.742460
3,-1.022355,-0.798086,-0.870479,-0.560857,-0.651187,-0.503974,-0.583043,-0.748179,-0.750450,-0.910898,...,-0.983420,0.065928,-0.907403,-0.501912,-0.168236,-0.157171,0.993769,-0.444816,-0.526385,0.742460
4,0.241888,1.550305,-0.389085,0.419916,-0.216914,0.155164,-0.001525,-0.556446,0.329427,0.115638,...,-0.983420,0.065928,0.203070,-0.501912,1.384715,-0.157171,-1.006270,-0.444816,1.899751,0.742460


In [362]:
scaled_df.describe().round(2)

,Income,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumWebPurchases,NumCatalogPurchases,...,Customer_Tenure,Children,Purchase_Frequency,Campaign_Engagement,Deal_Engagement,Education_Basic,Education_Graduation,Education_Master,Education_PhD,Household_Type_Partner
count,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,...,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00
mean,-0.00,-0.00,-0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,...,0.00,0.00,-0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-2.02,-1.70,-0.90,-0.66,-0.74,-0.69,-0.66,-0.84,-1.47,-0.91,...,-1.71,-1.26,-1.74,-0.50,-1.20,-0.16,-1.01,-0.44,-0.53,-1.35
25%,-0.67,-0.87,-0.83,-0.64,-0.67,-0.63,-0.63,-0.67,-0.75,-0.91,...,-0.80,-1.26,-0.91,-0.50,-0.69,-0.16,-1.01,-0.44,-0.53,-1.35
50%,-0.03,-0.00,-0.39,-0.46,-0.44,-0.47,-0.46,-0.38,-0.03,-0.23,...,0.01,0.07,-0.07,-0.50,-0.17,-0.16,0.99,-0.44,-0.53,0.74
75%,0.64,0.86,0.60,0.17,0.29,0.23,0.14,0.23,0.69,0.46,...,0.82,0.07,0.76,0.62,0.35,-0.16,0.99,-0.44,-0.53,0.74
max,24.55,1.72,3.53,4.34,6.90,4.06,5.72,6.10,8.25,8.67,...,1.73,2.73,2.70,5.11,6.56,6.36,0.99,2.25,1.90,0.74


### Interpretation 

All modelling features were successfully standardized using StandardScaler. Standardization ensures that variables measured on different scales contribute equally during clustering and prevents features with larger numerical ranges from dominating the distance calculations used by K-Means.

In [363]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

df.to_csv(
    "../data/processed/customer_features_final.csv",
    index=False
)

In [364]:
scaled_df.to_csv(
    "../data/processed/customer_clustering_ready.csv",
    index=False
)

#  Engineered Feature Summary

The following table summarizes the key engineered features created throughout this notebook and their intended business purpose.

In [365]:
feature_summary = pd.DataFrame({
    "Feature": [
        "Age",
        "Customer_Tenure",
        "Children",
        "Household_Type",
        "Total_Spending",
        "Purchase_Frequency",
        "Campaign_Engagement",
        "Deal_Engagement",
        "Customer_Value_Score"
    ],
    "Category": [
        "Customer",
        "Customer",
        "Customer",
        "Customer",
        "Behavioural",
        "Behavioural",
        "Behavioural",
        "Marketing",
        "Marketing"
    ],
    "Business Purpose": [
        "Represents customer age",
        "Measures relationship duration",
        "Captures household composition",
        "Represents household structure",
        "Measures total customer expenditure",
        "Measures purchasing activity",
        "Measures marketing responsiveness",
        "Measures promotional purchasing behaviour",
        "Summarizes overall customer value"
    ]
})

feature_summary

,Feature,Category,Business Purpose
0,Age,Customer,Represents customer age
1,Customer_Tenure,Customer,Measures relationship duration
2,Children,Customer,Captures household composition
3,Household_Type,Customer,Represents household structure
4,Total_Spending,Behavioural,Measures total customer expenditure
5,Purchase_Frequency,Behavioural,Measures purchasing activity
6,Campaign_Engagement,Behavioural,Measures marketing responsiveness
7,Deal_Engagement,Marketing,Measures promotional purchasing behaviour
8,Customer_Value_Score,Marketing,Summarizes overall customer value
